# Generate Replication Report (LaTeX)

**Run this notebook from the project root** so that `OUTPUT_DIR` and paths resolve correctly.

This notebook builds a **single LaTeX document** that satisfies the replication checklist:

- Describes the nature of the replication project
- Contains **all** tables and charts produced by the code (including Table 2 and the partial dependence figure)
- Gives a high-level overview of successes and challenges
- Explains data sources used
- **No code snippets** in the generated PDF—only narrative, tables, and figures

The generated file is written to **`reports/replication_report_generated.tex`**. Compile from the project root with:

```bash
cd reports && pdflatex replication_report_generated.tex
```

In [12]:
import sys
from pathlib import Path

# Project root = directory that contains "reports" and "src" (outermost reports)
_cwd = Path(".").resolve()
if (_cwd / "reports").is_dir():
    ROOT = _cwd
else:
    ROOT = _cwd.parent  # e.g. running from src/ -> project root

sys.path.insert(0, str(ROOT / "src"))
try:
    from settings import config
    OUTPUT_DIR = Path(config("OUTPUT_DIR"))
    IMAGES_DIR = Path(config("IMAGES_DIR"))
except Exception:
    OUTPUT_DIR = ROOT / "_output"
    IMAGES_DIR = OUTPUT_DIR / "images"

REPORTS_DIR = ROOT / "reports"  # outermost reports folder
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("REPORTS_DIR:", REPORTS_DIR)

OUTPUT_DIR: /Users/leanon/Desktop/FinMath/26Winter/Full-Stack_QF/p02_van_binsbergen_han_lopez-lira_2022/_output
REPORTS_DIR: /Users/leanon/Desktop/FinMath/26Winter/Full-Stack_QF/p02_van_binsbergen_han_lopez-lira_2022/src/reports


## 1. Build Table 2 (Term Structure) from CSV

Read `table2_term_structure.csv` (produced by `src/table2_term_structure.py`) and convert it to a LaTeX `tabular` so that the report is **automatically generated** from the code output.

In [13]:
import pandas as pd

table2_path = OUTPUT_DIR / "table2_term_structure.csv"
if not table2_path.exists():
    raise FileNotFoundError(f"Run the pipeline first to create {table2_path}")

df = pd.read_csv(table2_path)

def latex_cell(val):
    if val == "" or (isinstance(val, float) and pd.isna(val)):
        return ""
    if isinstance(val, (int, float)):
        if isinstance(val, int) and val >= 1000:
            return f"{val:,}".replace(",", "{\\,}")
        return str(val)
    return str(val)

cols = ["Horizon", "RF", "AF", "AE", "(RF-AE)", "(AF-AE)", "(RF-AE)^2", "(AF-AE)^2", "(AF-RF)/P", "N"]
header = " & ".join(c.replace("^2", "$^2$") if "^2" in c else c for c in cols) + " \\\\"
lines = ["\\toprule", header, "\\midrule"]

for _, row in df.iterrows():
    cells = [str(row["Horizon"])]
    for c in cols[1:]:
        cells.append(latex_cell(row.get(c, "")))
    line = " & ".join(cells) + " \\\\"
    if row["Horizon"] == "t-stat":
        line = "\\textit{t-stat} & " + " & ".join(cells[1:]) + " \\\\"
    lines.append(line)

lines.append("\\bottomrule")
table2_latex = "\n".join(lines)
print("Table 2 LaTeX fragment (first 5 lines):")
print("\n".join(lines[:5]))

Table 2 LaTeX fragment (first 5 lines):
\toprule
Horizon & RF & AF & AE & (RF-AE) & (AF-AE) & (RF-AE)$^2$ & (AF-AE)$^2$ & (AF-RF)/P & N \\
\midrule
One-quarter-ahead & 0.246 & 0.268 & 0.249 & -0.003 & 0.019 & 0.565 & 0.1 & 0.019 & 881903.0 \\
\textit{t-stat} &  &  &  & -1.64 & 6.16 &  &  & -13.92 &  \\


## 2. Assemble the full LaTeX document

Sections: Overview, Data Sources, Results (Table 2 + Partial Dependence Figure), Successes and Challenges. Figures and table are included by reference; the table body is generated from the CSV above.

In [14]:
preamble = r"""
\documentclass[11pt]{article}
\usepackage[utf8]{inputenc}
\usepackage{graphicx}
\usepackage{booktabs}
\usepackage{float}
\usepackage{caption}
\usepackage[margin=1in]{geometry}
\usepackage{hyperref}

\newcommand{\PathToOutput}{../_output}

\begin{document}
"""

overview = r"""
\title{Replication Report: Man vs.\ Machine Learning\\
{\large van Binsbergen, Han, Lopez-Lira (2022)}}
\author{Replication Project}
\date{\today}
\maketitle

\section{Project Overview}

This document reports the replication of the empirical analysis in van Binsbergen, Han, and Lopez-Lira (2022), ``Man vs.\ Machine Learning: The Term Structure of Earnings Expectations and Conditional Biases.'' The paper asks whether machine learning can produce better earnings forecasts than professional equity analysts and whether systematic biases in analyst forecasts can be explained by regulatory and market-structure variables.

The methodology uses a rolling-window \textbf{Random Forest} (RF) trained on firm-level financial ratios, macroeconomic indicators, and lagged earnings. For five forecast horizons (one to three quarters ahead, one and two years ahead), the RF forecast is compared to the consensus analyst forecast (IBES) and to subsequently realized EPS (CRSP). The key quantity is the price-scaled bias $(AF - RF)/P$, where $AF$ is the analyst forecast, $RF$ is the RF prediction, and $P$ is the stock price.
"""

data_sources = r"""
\section{Data Sources}

\textbf{WRDS / IBES.} Consensus analyst earnings forecasts and actual reported EPS from IBES (\texttt{ibes.statsum\_epsus}), US firms, forecast-period indicators for quarterly and annual horizons. Sample starts January 1985.

\textbf{WRDS / CRSP.} Daily stock data (prices, returns, split-adjustment factors) from \texttt{crsp.dsf} for NYSE, AMEX, NASDAQ. IBES--CRSP link via CUSIP and date-range overlap. Actual EPS is adjusted for splits using CRSP cumulative adjustment factors.

\textbf{WRDS / Financial Ratios.} Firm-level accounting features from \texttt{wrdsapps\_finratio\_ibes.firm\_ratio\_ibes}. Missing values imputed with within-period, within-industry medians (Fama--French 49) and forward-filled by firm.

\textbf{Philadelphia Fed.} Real-time macro data: real GDP, industrial production, real personal consumption, unemployment. Log growth rates and levels merged into the panel with an as-of backward merge on the estimate date to avoid look-ahead bias.
"""

results_intro = r"""
\section{Results}

\subsection{Table 2: Term Structure of Earnings Forecasts}

Table~\ref{tab:term_structure} reports time-series averages of RF, analyst forecast (AF), and realized EPS (AE), their differences, squared differences, and the price-scaled bias $(AF-RF)/P$, with Newey--West $t$-statistics (3 lags for quarterly, 12 for annual horizons). The RF forecast error $(RF-AE)$ is close to zero and statistically insignificant; the analyst error $(AF-AE)$ is positive and significant. The column $(AF-RF)/P$ is negative and significant at all horizons, indicating analysts systematically over-forecast relative to the machine.
"""

In [15]:
pdp_fig = r"""
\subsection{Figure: Partial Dependence of Realized EPS on Analyst Forecasts}

Figure~\ref{fig:pdp} shows the partial dependence of the RF-predicted realized EPS on the standardized consensus analyst forecast for the Q1 horizon (replicating Figure~1 of the paper). The relationship is non-linear: at high analyst forecasts the RF predicts a smaller increase in actual EPS, consistent with analyst over-optimism. The grey band is a 95\% confidence interval from ICE curves.

\begin{figure}[H]
  \centering
  \caption{Realized EPS as a Non-Linear Function of Analysts' Forecasts (Q1)}
  \label{fig:pdp}
  \includegraphics[width=0.85\linewidth]{\PathToOutput/images/partial_dependence_meanest.png}
\end{figure}
"""

successes_challenges = r"""
\section{Successes and Challenges}

\subsection{Successes}

The replication reproduces the main findings: (1)~The RF produces smaller mean forecast errors than the analyst consensus at all horizons; $(AF-RF)/P$ is negative and statistically significant (Table~\ref{tab:term_structure}). (2)~The partial dependence plot (Figure~\ref{fig:pdp}) recovers the paper's non-linear relationship. (3)~The pipeline from raw data to tables and figures is automated with \texttt{doit}.

\subsection{Challenges}

Data access requires an institutional WRDS subscription. Rolling-window RF training across five horizons is computationally intensive. Some data-cleaning details in the paper are under-specified (e.g., exact adjustment-factor alignment, industry imputation). Extending the sample beyond the paper's end date may change aggregate statistics.
"""

table2_full = (
    r"""
\begin{table}[H]
\centering
\caption{Term Structure of Earnings Forecasts via Machine Learning}
\label{tab:term_structure}
\small
\begin{tabular}{lccccccccc}
"""
    + table2_latex + "\n"
    + r"""
\end{tabular}
\end{table}
"""
)

In [16]:
full_latex = (
    preamble
    + overview
    + data_sources
    + results_intro
    + table2_full
    + pdp_fig
    + successes_challenges
    + "\n\\end{document}\n"
)

out_tex = REPORTS_DIR / "replication_report_generated.tex"
out_tex.write_text(full_latex, encoding="utf-8")
print(f"Written: {out_tex}")
print("To compile: cd reports && pdflatex replication_report_generated.tex")

Written: /Users/leanon/Desktop/FinMath/26Winter/Full-Stack_QF/p02_van_binsbergen_han_lopez-lira_2022/src/reports/replication_report_generated.tex
To compile: cd reports && pdflatex replication_report_generated.tex
